# Who Goes There: look at the data on Kaggle

A small notebook for looking at CCT20 photos (item 8.07): a grid of training photos per class,
day vs night, from the attached 512 px cache, and the five data charts of
`trailcam/data/datavis.py` with the night shares of the kept images, drawn with the cache's
day/night sidecar. It trains nothing and needs no GPU.

**Never commit this notebook with its outputs.** The photo grid shows dataset images: they stay
in the Kaggle session. Before a commit, Edit > Clear All Outputs (the data guard refuses a
notebook with any output, R04). Raw dataset images never go into `docs/figures/`; only the
charts of section 3, which are drawn from the metadata and hold no photo, may be copied there.

Only the train split's photos are shown: val, trans_val, cis_test and trans_test are not looked
at here (test-set isolation, R17).

## How to run on Kaggle

1. File > Import Notebook with its GitHub link
   (https://github.com/codx999/who-goes-there/blob/main/notebooks/kaggle_look_at_data.ipynb).
2. Add Input: the private 512 px cache dataset of item 6.03 (`cct20-512`). Accelerator: None.
   Internet: on (the install).
3. Set `COMMIT` to a `main` commit that holds `trailcam/data/datavis.py`, then run the cells one
   by one.
4. Section 3's PNGs in `/kaggle/working/figures/` should match the committed charts in
   `docs/figures/` (same metadata, sidecar and seed); nothing needs downloading.
5. Clear All Outputs before saving a version or committing.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: set COMMIT to a main commit that holds trailcam/data/datavis.py.
from pathlib import Path

COMMIT = "main"  # a full sha pins the code; "main" takes the latest
DATA_ROOT = Path("/kaggle/input/datasets/codx888/cct20-512/cct20_512")  # the 6.03 dataset
REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
SPLIT_SEED = 42  # the split of the data card and config.yaml
PER_CELL = 4  # photos per class for day and for night
FIGURES = Path("/kaggle/working/figures")  # section 3's charts

## 1. Install `trailcam` and load the metadata

The notebook exception of `CLAUDE.md`: `pip install` of `trailcam` from the public repo. The
metadata is loaded with the sidecar's night flags and split with `SPLIT_SEED`; counts only.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", f"git+{REPO_URL}@{COMMIT}"], check=True
)
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata  # noqa: E402
from trailcam.data.split import make_split  # noqa: E402

meta = load_metadata(DATA_ROOT / "annotations", DATA_ROOT / DAY_NIGHT_SIDECAR)
split = make_split(meta, SPLIT_SEED)
train = split.records["train"]
print(split)
print(f"train: {len(train)} images, {sum(bool(r.night) for r in train)} night")

## 2. Look at the data: training photos per class, day vs night

One row per class: `PER_CELL` day photos, then `PER_CELL` night (infrared) photos, drawn at random
with `SPLIT_SEED`. Titles show the class and day or night only, never a file name. These outputs
are for your eyes in this session: clear them before any commit.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

from trailcam.data.classes import CLASSES

rng = random.Random(SPLIT_SEED)
fig, axes = plt.subplots(
    len(CLASSES), 2 * PER_CELL, figsize=(2.0 * 2 * PER_CELL, 1.6 * len(CLASSES))
)
for row, name in enumerate(CLASSES):
    for col_offset, night in ((0, False), (PER_CELL, True)):
        pool = [r for r in train if r.label == row and r.night is night]
        picks = rng.sample(pool, min(PER_CELL, len(pool)))
        for col in range(PER_CELL):
            ax = axes[row][col_offset + col]
            ax.axis("off")
            if col < len(picks):
                with Image.open(DATA_ROOT / "images" / picks[col].file_name) as image:
                    ax.imshow(image.convert("RGB"))
                ax.set_title(f"{name}, {'night' if night else 'day'}", fontsize=8)
fig.suptitle("CCT20 train photos per class, day vs night (Beery et al. 2018, CDLA-Permissive 1.0)")
fig.tight_layout()
plt.show()

## 3. The data charts with the kept-image night shares

`python -m trailcam.data.datavis` with the sidecar: the same five charts as `docs/figures/`, but
the day/night chart counts the kept images of each split (train, val, trans_val, cis_test,
trans_test) instead of the cache build's numbers over all annotated images. The charts are drawn
from the metadata only, so they may be copied into `docs/figures/` (within the guard's budget).

In [ ]:
from trailcam.data import datavis

datavis.main(
    [
        "--root",
        str(DATA_ROOT / "annotations"),
        "--seed",
        str(SPLIT_SEED),
        "--sidecar",
        str(DATA_ROOT / DAY_NIGHT_SIDECAR),
        "--out",
        str(FIGURES),
    ]
)